# 🏥 RAG for Medical

## The Use Case

Medical professionals need fast access to:
- Clinical guidelines and protocols
- Drug interaction data
- Research summaries

RAG can surface the right guideline in seconds — without replacing clinical judgement.

## What's Special About Medical?

```
1. High stakes — wrong info can harm patients
2. Specialised vocabulary — medical terminology is precise
3. Evidence levels matter — RCT > case study
4. Dates matter — old guidelines may be superseded
5. Always: "Consult a physician" disclaimer
```

In [1]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')
print("Ready!")

Ready!


In [2]:
# Simulated clinical knowledge base
# Each entry includes an evidence level

medical_docs = [
    {"source": "AHA Guidelines 2023",       "evidence": "Level A", "text": "For patients with hypertension, first-line treatment includes ACE inhibitors or ARBs, particularly in those with diabetes or CKD."},
    {"source": "WHO Drug Reference 2024",    "evidence": "Level B", "text": "Ibuprofen is contraindicated in patients with active GI bleeding or renal impairment. Use with caution in elderly patients."},
    {"source": "CDC Diabetes Protocol 2023", "evidence": "Level A", "text": "HbA1c target for most non-pregnant adults with diabetes is less than 7%. Metformin remains the preferred initial pharmacologic agent."},
    {"source": "NICE Asthma Guidelines",     "evidence": "Level A", "text": "Short-acting beta-agonists (SABAs) are recommended for immediate relief of asthma symptoms. Inhaled corticosteroids are first-line preventer therapy."},
    {"source": "Drug Interaction DB v5",     "evidence": "Level B", "text": "Warfarin interacts significantly with NSAIDs, increasing bleeding risk. Monitor INR closely if co-administration is necessary."},
    {"source": "Screening Guidelines 2024",  "evidence": "Level B", "text": "Colorectal cancer screening is recommended for average-risk adults starting at age 45 using colonoscopy every 10 years or annual FIT."},
]

texts      = [d["text"] for d in medical_docs]
embeddings = model.encode(texts)
print(f"Indexed {len(medical_docs)} clinical guidelines")

Indexed 6 clinical guidelines


In [3]:
# Medical search — shows source AND evidence level

def medical_search(question, top_k=2):
    q_emb  = model.encode(question)
    scores = cosine_similarity([q_emb], embeddings)[0]
    top_idx = np.argsort(scores)[::-1][:top_k]

    print(f"Clinical Query: '{question}'")
    print("-" * 60)
    for rank, idx in enumerate(top_idx, 1):
        doc = medical_docs[idx]
        print(f"  {rank}. [{scores[idx]:.3f}] {doc['source']} | Evidence: {doc['evidence']}")
        print(f"     {doc['text']}")
    print()
    print("⚠️  For clinical use only. Always verify with current guidelines and consult a physician.")
    print()

medical_search("first-line treatment for high blood pressure")
medical_search("can I give ibuprofen to a patient with kidney problems?")
medical_search("diabetes medication target levels")

Clinical Query: 'first-line treatment for high blood pressure'
------------------------------------------------------------
  1. [0.722] AHA Guidelines 2023 | Evidence: Level A
     For patients with hypertension, first-line treatment includes ACE inhibitors or ARBs, particularly in those with diabetes or CKD.
  2. [0.245] NICE Asthma Guidelines | Evidence: Level A
     Short-acting beta-agonists (SABAs) are recommended for immediate relief of asthma symptoms. Inhaled corticosteroids are first-line preventer therapy.

⚠️  For clinical use only. Always verify with current guidelines and consult a physician.

Clinical Query: 'can I give ibuprofen to a patient with kidney problems?'
------------------------------------------------------------
  1. [0.678] WHO Drug Reference 2024 | Evidence: Level B
     Ibuprofen is contraindicated in patients with active GI bleeding or renal impairment. Use with caution in elderly patients.
  2. [0.317] AHA Guidelines 2023 | Evidence: Level A
     For pa

In [4]:
# Sort results by BOTH relevance AND evidence level

EVIDENCE_WEIGHT = {"Level A": 1.0, "Level B": 0.8, "Level C": 0.6, "Expert Opinion": 0.4}

def evidence_weighted_search(question, top_k=3):
    q_emb  = model.encode(question)
    scores = cosine_similarity([q_emb], embeddings)[0]

    # Blend semantic score with evidence level weight
    blended = []
    for i, doc in enumerate(medical_docs):
        ev_weight  = EVIDENCE_WEIGHT.get(doc["evidence"], 0.5)
        final_score = 0.7 * scores[i] + 0.3 * ev_weight
        blended.append((final_score, i))

    blended.sort(reverse=True)
    top_results = blended[:top_k]

    print(f"Evidence-weighted search: '{question}'")
    print("-" * 60)
    for score, idx in top_results:
        doc = medical_docs[idx]
        print(f"  [{score:.3f}] {doc['source']} ({doc['evidence']})")
        print(f"     {doc['text'][:80]}...")
    print()

evidence_weighted_search("blood pressure management")

Evidence-weighted search: 'blood pressure management'
------------------------------------------------------------
  [0.685] AHA Guidelines 2023 (Level A)
     For patients with hypertension, first-line treatment includes ACE inhibitors or ...
  [0.415] NICE Asthma Guidelines (Level A)
     Short-acting beta-agonists (SABAs) are recommended for immediate relief of asthm...
  [0.367] CDC Diabetes Protocol 2023 (Level A)
     HbA1c target for most non-pregnant adults with diabetes is less than 7%. Metform...



## Key Rules for Medical RAG

1. **Weight by evidence level** — Level A RCTs > expert opinion
2. **Always show the source + year** — guidelines get updated
3. **Mandatory disclaimer** — not a substitute for clinical judgement
4. **Restrict to your domain** — don't answer outside indexed guidelines
5. **Audit logs** — required for clinical compliance